In [1]:
# Add the path of the parent dir to the system path
import sys
sys.path.append('../')

In [2]:
from typing import Dict, List, Tuple, Generator, OrderedDict, Union, Optional
from pathlib import Path
import torch
import numpy as np
import cv2
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
import click
import pandas as pd

import json
import os
from PIL import Image
from einops import rearrange

from configs import g_conf, merge_with_yaml
from _utils.training_utils import check_saved_checkpoints
from _utils import utils
from dataloaders.transforms import canbus_normalization, ted_transform, decode_float_directions_to_str

from network.models.architectures.CIL_multiview.CIL_multiview import CIL_multiview

def load_model_from_checkpoint(model: CIL_multiview, 
                               checkpoint_path: str, 
                               checkpoint_number: int) -> CIL_multiview:
    checkpoint = check_saved_checkpoints(checkpoint_path, checkpoint_number)  # works even if checkpoint_number is None
    checkpoint = torch.load(checkpoint)

    new_state_dict = {}
    for k, v in checkpoint['model'].items():
        new_state_dict[k[7:]] = v

    new_state_dict = OrderedDict(new_state_dict)

    model.load_state_dict(new_state_dict)

    return model

def model_forward(
    model: CIL_multiview, 
    data: dict
) -> torch.Tensor:
    # Get the batch size
    cam_names = [c for c in g_conf.DATA_USED if 'rgb' in c]
    cam = len(cam_names)  # Number of cameras
    batch_size = data[cam_names[0]].shape[0]

    # Stack RGB images [batch, 3, h, w]
    src_images = [data[camera_type] for camera_type in g_conf.DATA_USED if 'rgb' in camera_type]
    x = torch.stack(src_images, dim=1)  # [batch, cam_num, 3, h, w]
    x = rearrange(x, 'B cam C H W -> (B cam) C H W')
    
    # Process directions
    d = data['cmd_fix_can_bus']['direction'].float()  # [batch, 4]
    # d = torch.tensor(directions, device='cuda').float().view(-1, 1)  # [batch, 1]
    
    # Process speeds
    s = data['cmd_fix_can_bus']['speed'].view(-1, 1)  # [batch, 1]
    
    # Forward pass
    e_p, _ = model.encoder_embedding_perception(x)
    
    e_p = rearrange(e_p, '(B cam) dim h w -> B (cam h w) dim', cam=cam)  # [B, S*cam*h*w, D]
    e_d = model.command(d).unsqueeze(1)
    e_s = model.speed(s).unsqueeze(1)
    
    if model.num_register_tokens > 0:
        e_p = torch.cat([model.register_tokens.repeat(batch_size, 1, 1), e_p], dim=1)
    
    e_p = e_p + e_d + e_s
    e_p = e_p + model.positional_encoding
    
    in_memory, _ = model.tx_encoder(e_p)

    # Get the action output (if no decoder is used, the sa and mha weights are None)
    action_output, _, _ = model.action_prediction(in_memory, cam)  # [B, 1, t=len(TARGETS)]

    return action_output  # [B, 1, len(TARGETS)]

In [3]:
import torch
import numpy as np
from typing import List, Tuple, Dict, Union
import time
from tqdm import tqdm
import pandas as pd

def create_synthetic_batch(image_shape: Tuple[int, int, int]) -> Dict[str, Union[torch.Tensor, Dict[str, torch.Tensor]]]:
    """Creates a synthetic batch of data with the specified image shape using camera names from config."""
    batch_data = {
        'cmd_fix_can_bus': {
            'direction': torch.tensor([[0., 1., 0., 0.]], device='cuda'),
            'speed': torch.tensor([[0.]], device='cuda')
        }
    }
    
    # Add synthetic data for each RGB camera from config
    rgb_cameras = [cam for cam in g_conf.DATA_USED if 'rgb' in cam.lower()]
    for camera in rgb_cameras:
        batch_data[camera] = torch.rand(1, *image_shape, device='cuda')
        
    return batch_data

def benchmark_model(model: torch.nn.Module, image_shape: Tuple[int, int, int],
                   num_iterations: int = 100, warmup_iterations: int = 10) -> Dict[str, float]:
    """Benchmark model performance for a specific image resolution."""
    model.eval()
    torch.cuda.empty_cache()
    batch_data = create_synthetic_batch(image_shape)
    
    with torch.no_grad():
        for _ in range(warmup_iterations):
            _ = model_forward(model, batch_data)
    
    times = []
    with torch.no_grad():
        for _ in tqdm(range(num_iterations), desc=f"Testing resolution {image_shape[1]}x{image_shape[2]}", leave=False):
            torch.cuda.synchronize()
            start_time = time.perf_counter()
            _ = model_forward(model, batch_data)
            torch.cuda.synchronize()
            end_time = time.perf_counter()
            times.append(end_time - start_time)
    
    times = np.array(times)
    fps = 1.0 / times
    return {
        'mean_fps': float(np.mean(fps)),
        'std_fps': float(np.std(fps)),
        'min_fps': float(np.min(fps)),
        'max_fps': float(np.max(fps)),
        'mean_latency_ms': float(np.mean(times) * 1000),
        'std_latency_ms': float(np.std(times) * 1000)
    }

def benchmark_resolutions(model: torch.nn.Module, resolution: List[Tuple[int, int, int]],
                         num_iterations: int = 100, warmup_iterations: int = 10) -> pd.DataFrame:
    """Benchmark model across different resolutions."""
    results = []
    c, height, width = resolution
    stats = benchmark_model(
        model=model,
        image_shape=resolution,
        num_iterations=num_iterations,
        warmup_iterations=warmup_iterations
    )
    results.append({
        'resolution': f"{width}x{height}",
        'width': width,
        'height': height,
        **stats
    })
    return pd.DataFrame(results)

def run_benchmark(model: CIL_multiview, resolution: List[Tuple[int, int, int]], 
                 num_iterations: int = 100, warmup_iterations: int = 10) -> pd.DataFrame:
    """Main function to run the benchmark suite."""
    model = model.cuda()
    results_df = benchmark_resolutions(
        model=model,
        resolution=resolution,
        num_iterations=num_iterations,
        warmup_iterations=warmup_iterations
    )
    return results_df

In [4]:
# Set CUDA device
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '2'
os.environ['TRAINING_RESULTS_ROOT'] = '/data/121-2/Experiments/dporres/VisionTFM'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'GPU: {torch.cuda.get_device_name()}')

GPU: NVIDIA A40


In [5]:
exp_batch = 'TED'
exp_name = '00_CIL++_3cam_Town01_17hdata-Filtered_noAttention_bs120_400x225'

checkpoint_number = 40

# Load model configuration and initialize model
merge_with_yaml(os.path.join('/data/121-2/Experiments/dporres/CILv2_multiview/', 'configs', exp_batch, f'{exp_name}.yaml'))
g_conf.PROCESS_NAME = 'train_val'
    
model = CIL_multiview(g_conf.MODEL_CONFIGURATION).to(device)
checkpoint_path = os.path.join(os.environ["TRAINING_RESULTS_ROOT"], '_results', g_conf.EXPERIMENT_BATCH_NAME, g_conf.EXPERIMENT_NAME, 'checkpoints')
model = load_model_from_checkpoint(model, checkpoint_path, checkpoint_number=checkpoint_number)

results = run_benchmark(
    model=model,
    resolution=g_conf.IMAGE_SHAPE,
    num_iterations=1000,
    warmup_iterations=10
)

# Print results
print("\nBenchmark Results:")
print(results.to_string(index=False))


Benchmark Results:
resolution  width  height    mean_fps   std_fps    min_fps     max_fps  mean_latency_ms  std_latency_ms
   400x225    400     225  143.912162  4.032112  74.267428  147.880469         6.956288        0.280878


In [6]:
exp_batch = 'TED'
exp_name = '03_CIL++_3cam_Town01_17hdata-Filtered_AttentionLossKL_AreaDownsampling_20xAttLoss_bs120_400x225'

checkpoint_number = 55

# Load model configuration and initialize model
merge_with_yaml(os.path.join('/data/121-2/Experiments/dporres/CILv2_multiview/', 'configs', exp_batch, f'{exp_name}.yaml'))
g_conf.PROCESS_NAME = 'train_val'
    
model = CIL_multiview(g_conf.MODEL_CONFIGURATION).to(device)
checkpoint_path = os.path.join(os.environ["TRAINING_RESULTS_ROOT"], '_results', g_conf.EXPERIMENT_BATCH_NAME, g_conf.EXPERIMENT_NAME, 'checkpoints')
model = load_model_from_checkpoint(model, checkpoint_path, checkpoint_number=checkpoint_number)

results = run_benchmark(
    model=model,
    resolution=g_conf.IMAGE_SHAPE,
    num_iterations=1000,
    warmup_iterations=10
)

# Print results
print("\nBenchmark Results:")
print(results.to_string(index=False))


Benchmark Results:
resolution  width  height    mean_fps   std_fps     min_fps     max_fps  mean_latency_ms  std_latency_ms
   400x225    400     225  144.956732  2.599587  104.254892  146.415927         6.901255         0.14868


In [7]:
exp_batch = 'TED'
exp_name = '02_CIL++_3cam_Town01_17hdata-Filtered_noAttention_bs60_960x540'

# checkpoint_number = 55

# Load model configuration and initialize model
merge_with_yaml(os.path.join('/data/121-2/Experiments/dporres/CILv2_multiview/', 'configs', exp_batch, f'{exp_name}.yaml'))
g_conf.PROCESS_NAME = 'train_val'
    
model = CIL_multiview(g_conf.MODEL_CONFIGURATION).to(device)
# checkpoint_path = os.path.join(os.environ["TRAINING_RESULTS_ROOT"], '_results', g_conf.EXPERIMENT_BATCH_NAME, g_conf.EXPERIMENT_NAME, 'checkpoints')
# model = load_model_from_checkpoint(model, checkpoint_path, checkpoint_number=checkpoint_number)

results = run_benchmark(
    model=model,
    resolution=g_conf.IMAGE_SHAPE,
    num_iterations=1000,
    warmup_iterations=10
)

# Print results
print("\nBenchmark Results:")
print(results.to_string(index=False))


Benchmark Results:
resolution  width  height   mean_fps   std_fps    min_fps    max_fps  mean_latency_ms  std_latency_ms
   960x540    960     540  59.341045  0.413821  57.943511  59.641542        16.852576        0.119488


In [12]:
exp_batch = 'TED'
exp_name = '03_CIL++_3cam_Town01_17hdata-Filtered_AttentionLossKL_AreaDownsampling_20xAttLoss_bs60_960x540'

# checkpoint_number = 50

# Load model configuration and initialize model
merge_with_yaml(os.path.join('/data/121-2/Experiments/dporres/CILv2_multiview/', 'configs', exp_batch, f'{exp_name}.yaml'))
g_conf.PROCESS_NAME = 'train_val'
    
model = CIL_multiview(g_conf.MODEL_CONFIGURATION).to(device)
# checkpoint_path = os.path.join(os.environ["TRAINING_RESULTS_ROOT"], '_results', g_conf.EXPERIMENT_BATCH_NAME, g_conf.EXPERIMENT_NAME, 'checkpoints')
# model = load_model_from_checkpoint(model, checkpoint_path, checkpoint_number=checkpoint_number)

results = run_benchmark(
    model=model,
    resolution=g_conf.IMAGE_SHAPE,
    num_iterations=1000,
    warmup_iterations=10
)

# Print results
print("\nBenchmark Results:")
print(results.to_string(index=False))


Benchmark Results:
resolution  width  height   mean_fps   std_fps    min_fps   max_fps  mean_latency_ms  std_latency_ms
   960x540    960     540  59.282304  0.484466  57.895807  59.64591        16.869583          0.1399


In [13]:
x = model.encoder_embedding_perception(torch.rand(1, 3, 960, 540).cuda())

In [15]:
x[0].shape

torch.Size([1, 512, 30, 17])

### Vanilla model

In [11]:
exp_batch = 'CILv2_recrexp'
exp_name = 'CILv2_3cam_Town01_2hdata1Weather_noAttention_bs120'

# checkpoint_number = 40

# Load model configuration and initialize model
merge_with_yaml(os.path.join('/data/121-2/Experiments/dporres/CILv2_multiview/', 'configs', exp_batch, f'{exp_name}.yaml'))
g_conf.PROCESS_NAME = 'train_val'
    
model = CIL_multiview(g_conf.MODEL_CONFIGURATION).to(device)
# checkpoint_path = os.path.join(os.environ["TRAINING_RESULTS_ROOT"], '_results', g_conf.EXPERIMENT_BATCH_NAME, g_conf.EXPERIMENT_NAME, 'checkpoints')
# model = load_model_from_checkpoint(model, checkpoint_path, checkpoint_number=checkpoint_number)

results = run_benchmark(
    model=model,
    resolution=g_conf.IMAGE_SHAPE,
    num_iterations=1000,
    warmup_iterations=10
)

# Print results
print("\nBenchmark Results:")
print(results.to_string(index=False))


Benchmark Results:
resolution  width  height    mean_fps   std_fps     min_fps     max_fps  mean_latency_ms  std_latency_ms
   300x300    300     300  144.227157  1.912236  135.518417  146.166691         6.934769        0.095174
